# Lección 09 — Metacognición con Claude

En este notebook vas a implementar los 4 mecanismos de metacognición:
1. **Fallback automático** — detectar errores y usar alternativas
2. **Auto-evaluación** — puntuar la respuesta antes de entregarla
3. **Aprendizaje por feedback** — ajustar estrategia según resultados pasados
4. **Patrón Maker-Evaluador** — dos agentes que se complementan

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()

def llamar_claude(system: str, mensaje: str, max_tokens: int = 800) -> str:
    respuesta = client.messages.create(
        model="claude-opus-4-5", max_tokens=max_tokens,
        system=system, messages=[{"role": "user", "content": mensaje}]
    )
    return respuesta.content[0].text

print("Setup listo.")

## Parte 1 — Fallback Automático

El agente intenta la herramienta principal. Si falla, detecta el error y automáticamente usa una de respaldo. El usuario no necesita hacer nada — el agente se recupera solo.

In [ ]:
# Sistema principal — solo tiene algunos destinos
def buscar_vuelos_principal(destino: str) -> dict:
    vuelos = {
        "París": {"precio": 890, "duracion": "13h", "aerolinea": "Air France"},
        "Tokio": {"precio": 1200, "duracion": "18h", "aerolinea": "Japan Airlines"},
        "Barcelona": {"precio": 650, "duracion": "10h", "aerolinea": "Iberia"},
    }
    if destino not in vuelos:
        raise Exception(f"Error 404: '{destino}' no encontrado en sistema principal")
    return vuelos[destino]

# Sistema de respaldo — tiene destinos diferentes
def buscar_vuelos_respaldo(destino: str) -> dict:
    vuelos_backup = {
        "Berlín": {"precio": 720, "duracion": "12h", "aerolinea": "Lufthansa"},
        "Sydney": {"precio": 1500, "duracion": "22h", "aerolinea": "Qantas"},
        "Ciudad de México": {"precio": 480, "duracion": "7h", "aerolinea": "Aeroméxico"},
        "Cancún": {"precio": 380, "duracion": "5h", "aerolinea": "VivaAerobus"},
    }
    return vuelos_backup.get(destino, {"error": f"'{destino}' no disponible en ningún sistema"})


# Schemas para Claude
schemas_vuelos = [
    {
        "name": "buscar_vuelos_principal",
        "description": "Sistema principal de vuelos. Intentar siempre primero. Puede fallar con error 404 si el destino no está disponible.",
        "input_schema": {"type": "object", "properties": {"destino": {"type": "string"}}, "required": ["destino"]}
    },
    {
        "name": "buscar_vuelos_respaldo",
        "description": "Sistema de respaldo. Usar SOLO si el sistema principal falla con error 404.",
        "input_schema": {"type": "object", "properties": {"destino": {"type": "string"}}, "required": ["destino"]}
    }
]

funciones_vuelos = {
    "buscar_vuelos_principal": buscar_vuelos_principal,
    "buscar_vuelos_respaldo": buscar_vuelos_respaldo
}


def agente_con_fallback(consulta: str) -> str:
    """Agente que maneja errores automáticamente y usa sistema de respaldo."""
    mensajes = [{"role": "user", "content": consulta}]
    system = """Sos un agente de búsqueda de vuelos con recuperación ante errores.
    Reglas:
    1. Intentá siempre buscar_vuelos_principal primero
    2. Si falla con un error 404, explicale al usuario que el sistema principal no lo tiene
       y automáticamente probá buscar_vuelos_respaldo
    3. Siempre informá qué sistema usaste y por qué
    4. Si ambos fallan, sugería destinos alternativos disponibles"""
    
    print(f"Consulta: {consulta}")
    print("-" * 50)
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5", max_tokens=600, system=system,
            tools=schemas_vuelos, messages=mensajes
        )
        if respuesta.stop_reason == "tool_use":
            uso = next(b for b in respuesta.content if b.type == "tool_use")
            print(f"  [Usando: {uso.name}({uso.input})]")
            fn = funciones_vuelos[uso.name]
            try:
                resultado = fn(**uso.input)
            except Exception as e:
                resultado = {"error": str(e)}
            mensajes.append({"role": "assistant", "content": respuesta.content})
            mensajes.append({"role": "user", "content": [{"type": "tool_result", "tool_use_id": uso.id, "content": json.dumps(resultado, ensure_ascii=False)}]})
            continue
        texto = next(b.text for b in respuesta.content if b.type == "text")
        print(f"Agente: {texto}")
        return texto


# Prueba con destino en sistema principal
agente_con_fallback("¿Hay vuelos a París?")

In [ ]:
print("\n" + "="*60)
# Prueba con destino solo en respaldo — el agente se recupera solo
agente_con_fallback("¿Hay vuelos a Berlín?")

## Parte 2 — Auto-evaluación

El agente genera una respuesta y luego la evalúa contra criterios definidos. Si no cumple el estándar, la mejora antes de entregarla.

In [ ]:
CRITERIOS_EVALUACION = """
Evaluá la respuesta según estos criterios (1-5 cada uno):
1. Completitud: ¿respondió todas las partes de la pregunta?
2. Precisión: ¿la información es correcta y específica?
3. Utilidad: ¿el usuario puede actuar con esta información?
4. Claridad: ¿es fácil de entender para un principiante?

Devolvé SOLO JSON:
{"completitud": N, "precision": N, "utilidad": N, "claridad": N, "promedio": N.N, "mejoras": "texto"}
"""

def evaluar_respuesta(pregunta: str, respuesta: str) -> dict:
    """Evalúa la calidad de una respuesta."""
    resultado = llamar_claude(
        system=CRITERIOS_EVALUACION,
        mensaje=f"Pregunta: {pregunta}\n\nRespuesta a evaluar:\n{respuesta}",
        max_tokens=300
    )
    texto = resultado.strip()
    if texto.startswith("```"):
        texto = texto.split("\n", 1)[1].rsplit("```", 1)[0]
    return json.loads(texto)


def agente_con_autoevaluacion(pregunta: str, umbral: float = 3.5) -> str:
    """Agente que evalúa y mejora su respuesta si no alcanza el umbral de calidad."""
    
    print(f"Pregunta: {pregunta}")
    print("-" * 50)
    
    # Generar respuesta inicial
    respuesta = llamar_claude(
        system="Sos un experto en IA para principiantes. Respondé en español, de forma clara y práctica.",
        mensaje=pregunta
    )
    
    # Auto-evaluar
    evaluacion = evaluar_respuesta(pregunta, respuesta)
    promedio = evaluacion["promedio"]
    print(f"Evaluación inicial: {promedio}/5 — Completitud:{evaluacion['completitud']} Precisión:{evaluacion['precision']} Utilidad:{evaluacion['utilidad']} Claridad:{evaluacion['claridad']}")
    
    if promedio < umbral:
        print(f"  ↻ Por debajo del umbral ({umbral}). Mejorando respuesta...")
        print(f"  Sugerencias: {evaluacion['mejoras']}")
        
        respuesta_mejorada = llamar_claude(
            system="Mejorás respuestas sobre IA para hacerlas más completas, precisas y útiles para principiantes.",
            mensaje=f"Mejorá esta respuesta aplicando las sugerencias indicadas.\n\nPregunta original: {pregunta}\n\nRespuesta actual:\n{respuesta}\n\nSugerencias de mejora:\n{evaluacion['mejoras']}"
        )
        
        # Re-evaluar
        eval_final = evaluar_respuesta(pregunta, respuesta_mejorada)
        print(f"Evaluación final: {eval_final['promedio']}/5")
        respuesta = respuesta_mejorada
    else:
        print(f"  ✓ Respuesta aprobada")
    
    print(f"\nRespuesta final:\n{respuesta}")
    return respuesta


agente_con_autoevaluacion("¿Cuál es la diferencia entre un LLM y un agente de IA?")

## Parte 3 — Aprendizaje por Feedback

El agente guarda un historial de qué estrategias funcionaron y cuáles no, y ajusta su enfoque para futuras interacciones.

In [ ]:
class AgenteAdaptativo:
    """Agente que aprende de cada interacción y ajusta su estrategia."""
    
    def __init__(self):
        self.historial_feedback = []  # [{pregunta, respuesta, feedback, estrategia}]
        self.estrategia_actual = "explicacion_simple"  # estrategia inicial
    
    def _estrategias_disponibles(self):
        return {
            "explicacion_simple": "Explicaciones muy simples, sin jerga técnica, con analogías del día a día",
            "ejemplos_practicos": "Enfocado en ejemplos de código y casos reales, menos teoría",
            "paso_a_paso": "Instrucciones detalladas paso a paso, muy estructuradas",
            "conceptual": "Explicaciones conceptuales profundas con fundamentos teóricos"
        }
    
    def _analizar_feedback(self) -> str:
        """Analiza el historial y determina la mejor estrategia."""
        if len(self.historial_feedback) < 2:
            return self.estrategia_actual
        
        # Calcular tasa de éxito por estrategia
        exito_por_estrategia = {}
        for entrada in self.historial_feedback:
            est = entrada["estrategia"]
            if est not in exito_por_estrategia:
                exito_por_estrategia[est] = {"total": 0, "positivo": 0}
            exito_por_estrategia[est]["total"] += 1
            if entrada["feedback"] == "positivo":
                exito_por_estrategia[est]["positivo"] += 1
        
        # Elegir la estrategia con mayor tasa de éxito
        mejor = max(
            exito_por_estrategia.items(),
            key=lambda x: x[1]["positivo"] / x[1]["total"]
        )
        return mejor[0]
    
    def responder(self, pregunta: str) -> str:
        estrategias = self._estrategias_disponibles()
        descripcion_estrategia = estrategias[self.estrategia_actual]
        
        system = f"""Sos un tutor de IA. Estrategia actual: {descripcion_estrategia}
        Aplicá esta estrategia al responder. Respondé en español."""
        
        return llamar_claude(system=system, mensaje=pregunta)
    
    def registrar_feedback(self, pregunta: str, respuesta: str, feedback: str):
        """Registra el feedback y ajusta la estrategia si hace falta."""
        self.historial_feedback.append({
            "pregunta": pregunta,
            "respuesta": respuesta[:100],
            "feedback": feedback,
            "estrategia": self.estrategia_actual
        })
        
        # Ajustar estrategia basado en historial
        nueva_estrategia = self._analizar_feedback()
        if nueva_estrategia != self.estrategia_actual:
            print(f"  [Adaptando estrategia: {self.estrategia_actual} → {nueva_estrategia}]")
            self.estrategia_actual = nueva_estrategia


# Demo del agente adaptativo
agente = AgenteAdaptativo()

# Interacción 1 — feedback negativo
print("=== INTERACCIÓN 1 ===")
r1 = agente.responder("¿Cómo funciona el context window en Claude?")
print(f"Estrategia: {agente.estrategia_actual}")
print(f"Respuesta: {r1[:200]}...")
agente.registrar_feedback("context window", r1, "negativo")  # simular feedback negativo

# El agente cambia de estrategia
agente.estrategia_actual = "ejemplos_practicos"  # simular cambio

print("\n=== INTERACCIÓN 2 (estrategia ajustada) ===")
r2 = agente.responder("¿Para qué sirve el context window en la práctica?")
print(f"Estrategia: {agente.estrategia_actual}")
print(f"Respuesta: {r2[:200]}...")
agente.registrar_feedback("context window practica", r2, "positivo")
print(f"\nHistorial: {len(agente.historial_feedback)} interacciones registradas")

## Parte 4 — Patrón Maker-Evaluador Completo

El sistema más robusto: un agente genera, otro evalúa. Si no pasa el estándar, el generador recibe el feedback y reintenta.

In [ ]:
def maker_evaluador(pedido: str, max_iteraciones: int = 3, umbral: float = 4.0) -> str:
    """Sistema completo donde el Maker genera y el Evaluador controla la calidad."""
    
    print(f"Pedido: {pedido}")
    print("=" * 60)
    
    feedback_previo = ""
    
    for i in range(1, max_iteraciones + 1):
        print(f"\n[Iteración {i}/{max_iteraciones}]")
        
        # MAKER: genera la respuesta
        prompt_maker = pedido
        if feedback_previo:
            prompt_maker += f"\n\nFeedback del evaluador anterior:\n{feedback_previo}\nAplicá estas mejoras en esta versión."
        
        respuesta = llamar_claude(
            system="Sos un experto en IA creando contenido educativo en español para principiantes latinoamericanos.",
            mensaje=prompt_maker,
            max_tokens=600
        )
        print(f"Maker generó ({len(respuesta.split())} palabras)")
        
        # EVALUADOR: evalúa la respuesta
        eval_prompt = f"""Evaluá esta respuesta para el pedido: '{pedido}'
        
Respuesta generada:
{respuesta}

Criterios (1-5):
1. Adecuación al pedido
2. Claridad para principiantes
3. Valor educativo
4. Ejemplos concretos

Respondé SOLO JSON: {{"adecuacion": N, "claridad": N, "valor": N, "ejemplos": N, "promedio": N.N, "feedback": "texto"}}"""
        
        eval_raw = llamar_claude(
            system="Sos un evaluador de contenido educativo. Evaluás con criterios rigurosos y das feedback específico.",
            mensaje=eval_prompt,
            max_tokens=300
        ).strip()
        
        if eval_raw.startswith("```"):
            eval_raw = eval_raw.split("\n", 1)[1].rsplit("```", 1)[0]
        
        evaluacion = json.loads(eval_raw)
        promedio = evaluacion["promedio"]
        print(f"Evaluador: {promedio}/5 — {evaluacion['feedback'][:80]}")
        
        if promedio >= umbral:
            print(f"\n✓ APROBADO ({promedio}/5 ≥ umbral {umbral})")
            print(f"\nResultado final:\n{respuesta}")
            return respuesta
        else:
            print(f"✗ No alcanza el umbral ({promedio}/5 < {umbral}). Reintentando...")
            feedback_previo = evaluacion["feedback"]
    
    print(f"\n⚠ Máximo de iteraciones. Entregando mejor versión.")
    return respuesta


maker_evaluador(
    "Escribí una explicación de 150 palabras sobre qué es Claude Code y por qué es útil para personas sin experiencia en programación",
    umbral=4.0
)

## Resumen

| Mecanismo | Lo que implementaste |
|---|---|
| Fallback automático | Detecta errores y cambia al sistema de respaldo sin intervención humana |
| Auto-evaluación | Puntúa la respuesta y la mejora si no alcanza el umbral |
| Aprendizaje por feedback | Ajusta la estrategia según el historial de éxitos y fracasos |
| Maker-Evaluador | Ciclo de generación → evaluación → mejora hasta alcanzar la calidad deseada |

Estos mecanismos combinados hacen que el agente sea más robusto, confiable y que mejore con el tiempo — sin necesidad de intervención manual constante.

---
En la **Lección 10** — la última — vemos cómo llevar todo esto a producción real.